# Week 1.4: Prompt Engineering — telling the model what you actually want
### Plain-English edition — no coding required

You have probably asked an AI something and got back a confident, useless answer.
That is almost never the model's fault. It is the instructions.

This notebook is about writing instructions a model can actually follow — and you
will build them with boxes, not code.

## How this notebook works
Every grey box below is a **step you run**, not code you read.

1. Some steps have **boxes to fill in** (type a word, drag a slider, tick a box).
2. Press the **▶ play button** on the left of the step.
3. Read the answer that appears underneath.

That's it. The computer writes and runs the code for you.

**Curious what it wrote?** Click **Show code** on any step to read it, then click
**Hide code** in the same place to tuck it away again.

> **Red text means something needs fixing** — read it and press play again.
> **Green means you got it right.** No red at all? You're good.

## What you will learn
- **System instructions** — setting the rules before the conversation starts
- **Zero-shot vs few-shot** — telling versus showing
- **Chain of thought** — making the model slow down and work
- **The prompt stack** — what really gets sent when you press enter

---
## Part 1: The system instruction

A system instruction is the standing brief you give a model before anyone talks to it:
who it is, what it knows, what it must not do, and how it should sound.

Fill in the four boxes and watch your brief assemble itself.

In [ ]:
#@title ▶ Step 1 — Write the rules for my digital twin { display-mode: "form" }
#@markdown **ROLE** — who is it pretending to be?
role = "a digital twin of Ama, a 17-year-old student from Accra" #@param {type:"string"}
#@markdown **KNOWLEDGE** — what is it allowed to know about?
knowledge = "Ama's daily routine, her school subjects, and her love of football" #@param {type:"string"}
#@markdown **CONSTRAINTS** — what must it never do?
constraints = "never invent facts about Ama, and never claim to be a real human" #@param {type:"string"}
#@markdown **TONE** — how should it sound?
tone = "friendly and direct, like a classmate" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

def box_print(title, body, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in body.split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def tokens_of(text):
    return max(1, int(len(text) / 4))

parts = {"ROLE": role, "KNOWLEDGE": knowledge, "CONSTRAINTS": constraints, "TONE": tone}
empty = [k for k, v in parts.items() if not v.strip()]
if empty:
    problem("These parts are still empty: " + ", ".join(empty))
    note("Fill in every box - each one does a different job - then press play again.")
else:
    brief = chr(10).join(k + ": " + v.strip() for k, v in parts.items())
    globals()["_SYSTEM"] = brief
    success("Your system instruction is ready.")
    print("")
    box_print("SYSTEM INSTRUCTION (sent before anyone types a word)", brief)
    print("")
    print("Roughly " + str(tokens_of(brief)) + " tokens. This is sent with EVERY message,")
    print("so a shorter brief is a cheaper and faster twin.")
    print("")
    print("So what? Each line does a job:")
    print("   ROLE        stops it answering as a generic assistant")
    print("   KNOWLEDGE   keeps it on topics it can actually speak to")
    print("   CONSTRAINTS is your safety rail - the most skipped and most important part")
    print("   TONE        decides whether it sounds like a friend or a form letter")

---
## Part 2: Telling vs showing

**Zero-shot** is asking with words alone. **Few-shot** is showing two or three worked
examples first. The second works far better for anything with a fixed format — and
costs you nothing but a few lines.

In [ ]:
#@title ▶ Step 2 — Compare asking with showing { display-mode: "form" }
#@markdown What do you want the model to do?
my_task = "Pull out the likes and dislikes from a sentence" #@param {type:"string"}
#@markdown Give it something to work on:
my_input = "I love football and jollof rice but I hate waking up early" #@param {type:"string"}
#@markdown Tick to add two worked examples (that is what makes it few-shot):
show_examples = True #@param {type:"boolean"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

def box_print(title, body, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in body.split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def tokens_of(text):
    return max(1, int(len(text) / 4))

zero = my_task.strip() + chr(10) + chr(10) + "Input: " + my_input.strip() + chr(10) + "Output:"
EX = ("Example 1" + chr(10) +
      "Input: I enjoy swimming but I cannot stand loud music" + chr(10) +
      'Output: {"likes": ["swimming"], "dislikes": ["loud music"]}' + chr(10) + chr(10) +
      "Example 2" + chr(10) +
      "Input: Reading is lovely, crowds are not" + chr(10) +
      'Output: {"likes": ["reading"], "dislikes": ["crowds"]}' + chr(10) + chr(10))
few = my_task.strip() + chr(10) + chr(10) + EX + "Now you:" + chr(10) + "Input: " + my_input.strip() + chr(10) + "Output:"
box_print("ZERO-SHOT - you only describe the task", zero)
print("   about " + str(tokens_of(zero)) + " tokens")
print("")
if show_examples:
    box_print("FEW-SHOT - you show the shape of the answer first", few)
    print("   about " + str(tokens_of(few)) + " tokens")
    print("")
    print("Same question. The second version answers something the first left open:")
    print("WHAT SHOULD THE ANSWER LOOK LIKE?")
    print("")
    print("With zero-shot you might get a chatty paragraph. With two examples")
    print("you reliably get the same tidy format every time, ready for your code.")
    print("")
    print("Cost of those examples: about " + str(tokens_of(few) - tokens_of(zero)) + " extra tokens. Usually worth it.")
else:
    note("Tick the box above to see the few-shot version beside it.")
print("")
print("So what? If the output format matters, show - do not tell.")

---
## Part 3: Making it slow down

Ask a model a multi-step question and it will often blurt out the first thing that
looks right. Adding four words — **"think step by step"** — makes it show its working,
and the working usually fixes the answer.

In [ ]:
#@title ▶ Step 3 — Watch "think step by step" change the answer { display-mode: "form" }
#@markdown A question that needs more than one step:
question = "Ama studies 2 hours on weekdays and 4 hours on Saturday. How many hours in a week?" #@param {type:"string"}
ask_it_to_think = True #@param {type:"boolean"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

def box_print(title, body, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in body.split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def tokens_of(text):
    return max(1, int(len(text) / 4))

if ask_it_to_think:
    prompt = question.strip() + chr(10) + chr(10) + "Think step by step, then give the answer."
else:
    prompt = question.strip() + chr(10) + chr(10) + "Answer:"
box_print("WHAT YOU SEND", prompt)
print("")
print("A typical reply looks like this:")
print("")
if ask_it_to_think:
    box_print("REPLY (asked to think)",
        "Step 1: weekdays are Monday to Friday, which is 5 days." + chr(10) +
        "Step 2: 5 days x 2 hours = 10 hours." + chr(10) +
        "Step 3: Saturday adds 4 hours." + chr(10) +
        "Step 4: 10 + 4 = 14." + chr(10) + chr(10) + "Answer: 14 hours.")
    print("")
    success("Correct - and you can see exactly where the number came from.")
    print("")
    print("So what? Those middle sentences are not decoration. The model")
    print("reads its own working as it writes, so each step steadies the next.")
else:
    box_print("REPLY (not asked to think)", "Answer: about 12 hours.")
    print("")
    problem("Wrong - and it sounds just as confident as the right answer.")
    print("")
    note("Tick the box above and run it again to see the difference.")
print("")
print("The examples above are illustrations of the typical behaviour.")
note("In Universal API Components you can send these for real and compare the actual replies.")

---
## Part 4: What actually gets sent

When someone types one line to your twin, the model receives far more than that line.
This is the full parcel.

In [ ]:
#@title ▶ Step 4 — Show me the whole parcel that gets sent { display-mode: "form" }
#@markdown What the person just typed:
new_message = "What should I do this evening?" #@param {type:"string"}
#@markdown How many earlier turns of the conversation to carry along:
history_turns = 2 #@param {type:"slider", min:0, max:6, step:1}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

def box_print(title, body, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in body.split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def tokens_of(text):
    return max(1, int(len(text) / 4))

brief = globals().get("_SYSTEM")
if not brief:
    brief = ("ROLE: a digital twin of Ama, a 17-year-old student from Accra" + chr(10) +
             "KNOWLEDGE: Ama's daily routine, her school subjects, and her love of football" + chr(10) +
             "CONSTRAINTS: never invent facts about Ama, and never claim to be a real human" + chr(10) +
             "TONE: friendly and direct, like a classmate")
    note("Using an example brief. Run Step 1 to use your own.")
    print("")
CHAT = [("Ama", "Hi, who are you?"), ("Twin", "I am your digital twin. I know your routine."),
        ("Ama", "Did I train this week?"), ("Twin", "You played football on Tuesday and Thursday."),
        ("Ama", "What about school?"), ("Twin", "You have maths and science tomorrow.")]
turns = CHAT[:int(history_turns) * 2]
print("THE PARCEL, IN ORDER:")
print("")
box_print("1. SYSTEM - the standing rules (every single time)", brief)
if turns:
    hist = chr(10).join(who + ": " + what for who, what in turns)
    print("")
    box_print("2. HISTORY - what was already said", hist)
else:
    hist = ""
    print("")
    note("No history carried. The twin will have forgotten everything said before.")
print("")
box_print("3. THE NEW MESSAGE", new_message.strip())
total = tokens_of(brief) + tokens_of(hist) + tokens_of(new_message)
print("")
print("Total sent: roughly " + str(total) + " tokens, for a message of " +
      str(tokens_of(new_message)) + " tokens.")
print("")
print("So what? The model has no memory of its own.")
print("It is handed the rules and the whole conversation again on every single turn,")
print("reads the lot, and predicts what comes next. That is the entire illusion of memory.")
if int(history_turns) >= 5:
    note("Notice how fast this grows. Long chats cost more and eventually hit the limit -")
    note("which is exactly the problem Week 2 solves.")

### 🎯 Exercise: grade your own prompt

Write a prompt and get it marked. The checks below are the ones that matter most
in practice — be honest, and rewrite until the red disappears.

In [ ]:
#@title ▶ Step 5 — Mark my prompt and tell me what is missing { display-mode: "form" }
#@markdown Write a prompt you would really use. Try a weak one first, then improve it.
my_prompt = "Tell me about my week" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

def box_print(title, body, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in body.split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def tokens_of(text):
    return max(1, int(len(text) / 4))

p = my_prompt.strip(); low = p.lower()
if len(p) < 5:
    problem("Please write a prompt first.")
else:
    checks = []
    checks.append(("Says who the model should be",
        any(k in low for k in ["you are", "act as", "role", "pretend", "as a", "twin"]),
        "Start with who it is: 'You are a digital twin of...'"))
    checks.append(("Says what to produce",
        any(k in low for k in ["list", "summar", "explain", "write", "predict", "extract", "compare", "suggest", "plan"]),
        "Name the output: list, summary, plan, explanation..."))
    checks.append(("Gives something concrete to work with",
        any(ch.isdigit() for ch in p) or len(p.split()) > 18,
        "Add the actual details - dates, numbers, the text to work on"))
    checks.append(("Sets a limit or a format",
        any(k in low for k in ["json", "bullet", "sentence", "word", "max", "under", "three", "two", "five", "short", "format", "table"]),
        "Say how long or what shape: 'three bullets', 'one sentence', 'as JSON'"))
    checks.append(("Says what NOT to do",
        any(k in low for k in ["do not", "don't", "never", "avoid", "without", "only"]),
        "Add a boundary: 'do not invent anything you were not told'"))
    passed = sum(1 for _, ok, _ in checks if ok)
    box_print("YOUR PROMPT", p)
    print("")
    print("Marking it against five things that matter:")
    print("")
    for name, ok, _ in checks:
        (success if ok else problem)(("PASS  " if ok else "MISS  ") + name)
    print("")
    print("Score: " + str(passed) + " out of 5")
    print("")
    misses = [fix for name, ok, fix in checks if not ok]
    if misses:
        print("To improve it:")
        for m in misses: print("   - " + m)
        print("")
        note("Rewrite the prompt above and press play again. Aim for no red.")
    else:
        success("Nothing missing. That is a prompt a model can actually follow.")
    print("")
    print("So what? A vague prompt does not get a vague answer.")
    print("It gets a confident, specific answer to a question you never asked.")

---
## Summary & Key Takeaways

1. **System instruction** — role, knowledge, constraints, tone. Constraints are the part people skip.
2. **Few-shot beats zero-shot** whenever the shape of the answer matters. Show, don't tell.
3. **"Think step by step"** turns a guess into working you can check.
4. **The prompt stack** — rules + history + message, re-sent every single turn. The model has no memory of its own.
5. **Grade your prompts** — who, what, with which details, in what shape, and what to avoid.

> Prompting is not a magic phrase. It is writing a brief clear enough that
> a very fast, very literal assistant cannot misread it.

### Next up
Sending these prompts to a real model over the internet — **Universal API Components**.